# Qwen 14B Inference + DOCX Export (Drive-loaded)
This notebook mounts Google Drive, loads your saved LoRA adapter from Drive, runs chunked span inference, and exports a cut-card `.docx`.

In [ ]:
!pip install -q unsloth python-docx
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch, json, re
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_COLOR_INDEX
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

In [ ]:
# CHANGE THIS to the folder you copied to Drive
MODEL_PATH = "/content/drive/MyDrive/qwen14b_span_prototype_filtered"
MAX_SEQ_LENGTH = 2048

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")
FastLanguageModel.for_inference(model)

# Defensive cleanup
model.generation_config.max_length = None
model.generation_config.max_new_tokens = None

print("Inference model loaded from:", MODEL_PATH)
print("max_length:", model.generation_config.max_length)
print("max_new_tokens:", model.generation_config.max_new_tokens)

In [ ]:
SYSTEM_PROMPT = """Extract formatting spans from debate card text.

Debate card context:
- The input is the text body of a debate evidence card.
- Formatting marks argumentative importance.

Span types:
- "ul" = underlined text
- "h" = highlighted text
- "e" = emphasized text

Output format:
Return only a compact JSON array of triples:
[[start, end, "type"], ...]

Rules:
- Offsets refer to the exact plain text shown by the user.
- start is inclusive, end is exclusive.
- Count spaces, punctuation, and newline characters.
- Do not rewrite or explain anything.
- Do not output markdown or code fences.
- If there are no spans, return [].
"""

In [ ]:
# =========================
# INFERENCE + CHUNKING + DIRECT DOCX RENDER FROM SPANS
# =========================

import json
import re
import ast
import torch
from docx import Document
from docx.shared import Pt
from docx.enum.text import WD_COLOR_INDEX
from unsloth import FastLanguageModel


def split_text_into_chunks(text, chunk_token_budget=1400):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    paragraphs = text.split("\n")

    chunks = []
    current_parts = []
    current_start = 0
    cursor = 0

    def token_len(s):
        return len(tokenizer(s, add_special_tokens=False)["input_ids"])

    def flush():
        nonlocal current_parts, current_start
        if current_parts:
            chunk_text = "\n".join(current_parts).strip()
            if chunk_text:
                chunks.append({"text": chunk_text, "start_char": current_start})
        current_parts = []

    for para in paragraphs:
        original_para = para
        para_with_nl = original_para + "\n"

        if not current_parts:
            current_start = cursor

        candidate = "\n".join(current_parts + [original_para]).strip()

        if candidate and token_len(candidate) <= chunk_token_budget:
            current_parts.append(original_para)
            cursor += len(para_with_nl)
            continue

        flush()

        if original_para.strip() and token_len(original_para) > chunk_token_budget:
            pieces = re.split(r"(?<=[.!?])\s+", original_para)
            piece_parts = []
            piece_start = cursor
            running_offset = 0

            for piece in pieces:
                test = (" ".join(piece_parts + [piece])).strip()
                if test and token_len(test) <= chunk_token_budget:
                    piece_parts.append(piece)
                else:
                    if piece_parts:
                        chunk_text = " ".join(piece_parts).strip()
                        chunks.append({"text": chunk_text, "start_char": piece_start})
                        piece_start = cursor + running_offset
                    piece_parts = [piece]
                running_offset += len(piece) + 1

            if piece_parts:
                chunk_text = " ".join(piece_parts).strip()
                chunks.append({"text": chunk_text, "start_char": piece_start})

            cursor += len(para_with_nl)
        else:
            current_parts = [original_para]
            current_start = cursor
            cursor += len(para_with_nl)

    flush()
    return chunks


def extract_json_array(text):
    """
    Parse a JSON array out of raw LLM output.
    Handles: curly/smart quotes, backtick code fences, extra prose,
    truncated output (missing closing brackets), single-quoted strings,
    trailing commas, prime characters used as quotes.
    """
    if not text or not text.strip():
        return []

    # 1. Strip markdown code fences
    text = re.sub(r"```[a-zA-Z]*\n?", "", text).strip()

    # 2. Normalize ALL quote variants to straight ASCII
    text = (text
        .replace("\u2018", "'").replace("\u2019", "'")   # ‘ ’
        .replace("\u201c", '"').replace("\u201d", '"')   # “ ”
        .replace("\u2032", "'").replace("\u2033", '"')   # prime / double-prime
        .replace("\u0060", "'")                            # backtick
    )

    # 3. Try direct parse
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    # 4. Extract [first-bracket .. last-bracket] substring
    start = text.find("[")
    end   = text.rfind("]")
    if start != -1 and end > start:
        candidate = text[start:end + 1]

        # 4a. json.loads
        try:
            obj = json.loads(candidate)
            if isinstance(obj, list):
                return obj
        except Exception:
            pass

        # 4b. ast.literal_eval (handles single-quoted strings)
        try:
            obj = ast.literal_eval(candidate)
            if isinstance(obj, list):
                return obj
        except Exception:
            pass

        # 4c. Salvage truncated output — close unclosed brackets
        opens = candidate.count("[") - candidate.count("]")
        if opens > 0:
            salvaged = candidate + "]" * opens
            for fn in (json.loads, ast.literal_eval):
                try:
                    obj = fn(salvaged)
                    if isinstance(obj, list):
                        return obj
                except Exception:
                    pass

    # 5. Last resort: strip trailing commas then retry
    cleaned = re.sub(r",\s*([}\]])", r"\1", text)
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    return []


def _merge_same_type_spans(spans, max_gap=0):
    if not spans:
        return []
    spans = sorted(spans, key=lambda x: (x[2], x[0], x[1]))
    merged = []
    cur_s, cur_e, cur_t = spans[0]
    for s, e, t in spans[1:]:
        if t == cur_t and s <= cur_e + max_gap:
            cur_e = max(cur_e, e)
        else:
            merged.append([cur_s, cur_e, cur_t])
            cur_s, cur_e, cur_t = s, e, t
    merged.append([cur_s, cur_e, cur_t])
    return merged


def _expand_to_word_boundaries(text, s, e):
    while s > 0 and not text[s - 1].isspace():
        s -= 1
    while e < len(text) and not text[e].isspace():
        e += 1
    return s, e


def _promote_and_expand(spans, text):
    ul_spans = [[s, e, t] for s, e, t in spans if t == "ul"]
    h_spans  = [[s, e, t] for s, e, t in spans if t == "h"]
    e_spans  = [[s, e, t] for s, e, t in spans if t == "e"]

    expanded_h = []
    for s, e, _ in h_spans:
        ns, ne = _expand_to_word_boundaries(text, s, e)
        expanded_h.append([ns, ne, "h"])
    h_spans = expanded_h

    expanded_e = []
    for s, e, _ in e_spans:
        ns, ne = _expand_to_word_boundaries(text, s, e)
        expanded_e.append([ns, ne, "e"])
    e_spans = expanded_e

    for s, e, _ in e_spans:
        if not any(hs <= s and e <= he for hs, he, _ in h_spans):
            h_spans.append([s, e, "h"])

    for s, e, _ in h_spans + e_spans:
        if not any(us <= s and e <= ue for us, ue, _ in ul_spans):
            ul_spans.append([s, e, "ul"])

    repaired = ul_spans + h_spans + e_spans
    repaired = _merge_same_type_spans(repaired, max_gap=0)
    repaired.sort(key=lambda x: (x[0], x[1], x[2]))
    return repaired


def clean_predicted_spans(spans, text):
    text_len = len(text)
    cleaned = []
    for item in spans:
        if not isinstance(item, (list, tuple)) or len(item) != 3:
            continue
        s, e, t = item
        try:
            s = int(s)
            e = int(e)
        except Exception:
            continue
        if t not in {"ul", "h", "e"}:
            continue
        s = max(0, min(s, text_len))
        e = max(0, min(e, text_len))
        if e > s:
            cleaned.append([s, e, t])
    cleaned = _merge_same_type_spans(cleaned, max_gap=0)
    cleaned = _promote_and_expand(cleaned, text)
    return cleaned


def merge_chunk_spans(chunks, chunk_spans):
    merged = []
    for chunk, spans in zip(chunks, chunk_spans):
        base = chunk["start_char"]
        for s, e, t in spans:
            merged.append([base + s, base + e, t])
    merged.sort(key=lambda x: (x[0], x[1], x[2]))
    return merged


def predict_spans_for_chunk(tag, text_chunk, max_input_tokens=2048, max_new_tokens=256):
    prompt = f"Tag: {tag}\nText:\n{text_chunk}\n\nOutput spans:"
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": prompt},
    ]

    rendered = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        rendered, return_tensors="pt", truncation=True, max_length=max_input_tokens
    ).to(model.device)

    FastLanguageModel.for_inference(model)
    model.generation_config.max_length = None
    model.generation_config.max_new_tokens = None
    model.eval()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    parsed  = extract_json_array(raw_text)
    cleaned = clean_predicted_spans(parsed, text_chunk)

    print("Raw model output (first 500 chars):", raw_text[:500])
    print("Raw parsed span count:", len(parsed))
    print("Cleaned span count:", len(cleaned))

    return raw_text, cleaned


def render_docx_from_spans(tag, cite, text_body, spans, output_path):
    text  = text_body.replace("\r\n", "\n").replace("\r", "\n")
    spans = sorted(spans, key=lambda x: (x[0], x[1], x[2]))

    boundary_points = {0, len(text)}
    for s, e, _ in spans:
        boundary_points.add(s)
        boundary_points.add(e)
    cuts = sorted(boundary_points)

    segs = []
    for a, b in zip(cuts[:-1], cuts[1:]):
        if b <= a:
            continue
        piece = text[a:b]
        if not piece:
            continue
        active = {t for s, e, t in spans if s <= a and b <= e}
        segs.append((piece, active))

    doc = Document()
    normal = doc.styles["Normal"]
    normal.font.name = "Times New Roman"
    normal.font.size = Pt(12)

    p_tag = doc.add_paragraph()
    r = p_tag.add_run(tag)
    r.bold = True
    r.font.name = "Times New Roman"
    r.font.size = Pt(12)

    p_cite = doc.add_paragraph()
    r = p_cite.add_run(cite)
    r.font.name = "Times New Roman"
    r.font.size = Pt(12)

    p_body = doc.add_paragraph()
    for piece, active in segs:
        run = p_body.add_run(piece)
        run.font.name = "Times New Roman"
        run.font.size = Pt(12)
        if "ul" in active:
            run.underline = True
        if "h" in active:
            run.font.highlight_color = WD_COLOR_INDEX.YELLOW
        if "e" in active:
            run.bold = True

    doc.save(output_path)
    return output_path


def predict_card_and_export_docx(tag, cite, text_body,
                                  output_path="/content/generated_cut_card.docx",
                                  chunk_token_budget=1400):
    full_text   = text_body.replace("\r\n", "\n").replace("\r", "\n").strip()
    chunks      = split_text_into_chunks(full_text, chunk_token_budget=chunk_token_budget)
    raw_generations = []
    chunk_spans = []

    for chunk in chunks:
        raw_text, spans = predict_spans_for_chunk(tag, chunk["text"])
        raw_generations.append(raw_text)
        chunk_spans.append(spans)

    merged_spans = merge_chunk_spans(chunks, chunk_spans)
    render_docx_from_spans(tag, cite, full_text, merged_spans, output_path)

    return {
        "chunks":          chunks,
        "raw_generations": raw_generations,
        "chunk_spans":     chunk_spans,
        "merged_spans":    merged_spans,
        "output_path":     output_path,
    }


In [ ]:
# Cell 7 has been removed.
# It was a broken duplicate of predict_card_and_export_docx that called
# undefined functions (spans_to_tagged_text, add_tagged_text_to_doc).
# The complete working pipeline is in Cell 6 above.
# Run order: Cell 1 → 2 → 3 → 4 → 5 → 6 → 8
print("Pipeline ready. Proceed to Cell 8.")


In [ ]:
# =========================
# INTERACTIVE TEST CELL
# =========================

tag = input("Enter tag: ").strip()
cite = input("Enter cite: ").strip()

print("Paste text body. Finish with a line containing only END.")
lines = []
while True:
    line = input()
    if line == "END":
        break
    lines.append(line)
text_body = "\n".join(lines).strip()

result = predict_card_and_export_docx(
    tag=tag,
    cite=cite,
    text_body=text_body,
    output_path="/content/generated_cut_card.docx",
    chunk_token_budget=1400,
)

print("Chunks used:", len(result["chunks"]))
for i, chunk in enumerate(result["chunks"], start=1):
    print(f"Chunk {i}: start_char={chunk['start_char']} len={len(chunk['text'])}")
    print("Raw model output:", result["raw_generations"][i-1][:300], "...")
    print()

print("Merged spans:", result["merged_spans"][:20], "..." if len(result["merged_spans"]) > 20 else "")
print("DOCX saved to:", result["output_path"])